# MASHeart: data exploration
An English, development-only companion notebook. Explore the real scientific-text data before training;
preserve the original official test and keep source groups together. Run top to bottom with Shift+Enter.

## Step 01: Restore the standalone project

**Input:** an embedded ZIP containing English Python modules, configuration, source data and license notices.
**Output:** `PROJECT_ROOT`, independent of the original research repository.
SHA-256 verifies byte identity; safe path checks keep every extracted file inside the runtime folder.
The long Base64 string is transport, not a learned model. Existing project files are used when opened
inside a complete MASHeart folder; the embedded copy makes a single uploaded notebook sufficient.

In [1]:
#@title Restore the verified MASHeart source/data bundle (collapse this technical cell)
ASSET_BASE64 = '...'
ASSET_SHA256 = '74675a45c395ef07b14257fc4bc5a0efa2abdd85e05155f3ff67db11908abfd6'
ASSET_FILES = {'configs/cpu.json': 'a71a4c63b8d44d38757b322b2c8fd79521311abb58f5dd728af27eb509e09e1e', 'configs/t4.json': '60a74ac9e275bd23dc8b850b432ad9abe697176bf19ec3b5d8da1bae604681c5', 'data/DATA_CARD.md': '46a4808ea035837c4aa8790a4dffbc3c2766d515747cc3c9da2d900eb1c0e324', 'data/manifest.json': '066a7905db66023fcd525aafecd91c552b0db955ad183661edcaec90888b92e1', 'data/raw/a_dev.jsonl': '289ec05d947a7f14463ee3508d57c57354f7c770580eb0b6cfebb31c051d10fa', 'data/raw/a_test.jsonl': '211d7e686ab3d7eb16a67a5e1d2e899c6fede10f3631b61cf827ad17c520230c', 'data/raw/bioclaim_source_provenance.json': '5ec8116fdd98e2cbd4aca01b5244b20d4722769e13a675772e995311e5873d5d', 'data/raw/pcc_source_provenance.json': 'a3bfdaa091938b51ad21866d6c4207418a2e70adb006aa9b2c2867942ad59311', 'data/raw/source_split_manifest.json': '661d86dcad76b5ba617a1d2b57c5a3d93843ef3e5123bf23b69a168ca7d516c0', 'LICENSE_DATA.md': 'dcc9a97c841aa1656757f15b1ef072452f787db11c4b90e1ce9ce4e08c437a3e', 'masheart/__init__.py': '1f26213d7e7052f4348ea6ced9541d13e369bba03c7d8a99bb08924de3868f70', 'masheart/__main__.py': 'a657f1e95cee82e003f657d739544ac112b3dd8618801a648c20bf1c3ab43e90', 'masheart/baseline.py': '97c8f27a60bb131638ca8ab34aa7b3f82f7fc277b185c72544e5e9c08e9e4b98', 'masheart/cli.py': 'b7c1474fe6f0c6b852b0a5221c1da334c22f93fdbae85713a7a4b33469ae9b6a', 'masheart/data.py': '561d01b6b95d49784b7bbea55b0bb11915db6f3101de3ddd9390c57be2569e9a', 'masheart/experiment.py': '9fe31066b2dacd9f90c407123755297b7c24b22dbe31e10327d71a773b3aca7b', 'masheart/gpu.py': '1c54524f2f9178d9de9712d90bd856383d91e5fb48b6ba30b4153d423a1e1ab4', 'masheart/metrics.py': '5bed4b4a20fed40966ebbd9be2d4157d9e5d656fbb873f04e825656406730e86', 'masheart/visualization.py': '7e61c66d3667109dde29964dd9605470fcfa8c1035d9eef8dee049ce5f308623', 'requirements-t4.txt': '6e81265ccbbb3d20f975444c2d1f2ace62fa984b109c7155a1a8a70d31c4d65b', 'requirements.txt': '5ad13c5c88e2cd254921260ffbe73f68aa44723f5296981825258f23b986f779'}

import base64, hashlib, io, json, sys, zipfile
from pathlib import Path

candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p for p in candidates if (p / 'masheart' / 'data.py').is_file()
                     and (p / 'data' / 'manifest.json').is_file()), None)
if PROJECT_ROOT is None:
    raw_archive = base64.b64decode(ASSET_BASE64)
    assert hashlib.sha256(raw_archive).hexdigest() == ASSET_SHA256
    PROJECT_ROOT = Path.cwd() / ('MASHeart_runtime_' + ASSET_SHA256[:10])
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw_archive)) as archive:
        for member in archive.infolist():
            target = (PROJECT_ROOT / member.filename).resolve()
            if not target.is_relative_to(PROJECT_ROOT.resolve()) or member.filename not in ASSET_FILES:
                raise ValueError('Unexpected archive path; extraction stopped.')
            content = archive.read(member)
            assert hashlib.sha256(content).hexdigest() == ASSET_FILES[member.filename]
            if target.exists() and target.read_bytes() != content:
                raise RuntimeError('An embedded source file was changed. Use a fresh runtime directory.')
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)
PROJECT_ROOT = PROJECT_ROOT.resolve()
sys.path.insert(0, str(PROJECT_ROOT))
print('Project:', PROJECT_ROOT)
print('No parent HeartMAS checkout is required.')


Project: C:\Users\huynh\OneDrive\Máy tính\ThsHUIT\HeartMAS\MASHeart\.verification\01_Data_Exploration_runtime\MASHeart_runtime_74675a45c3
No parent HeartMAS checkout is required.


## Step 02: Install the measured CPU environment

`importlib.metadata.version()` reads actual installed versions. The supplied requirements record the
reference environment. Only missing/different packages are installed. If an older module is already in
memory, restart after installation; pip cannot replace objects already imported into a kernel.
The local verification host already has these packages and performs no installation or network request.

In [2]:
import importlib.metadata, importlib.util, subprocess

required = {}
for line in (PROJECT_ROOT / 'requirements.txt').read_text().splitlines():
    if line.strip() and not line.startswith('#'):
        name, version = line.strip().split('==')
        required[name] = version
needed, loaded_changes = [], []
for name, version in required.items():
    try:
        installed = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        needed.append(name + '==' + version)
        module = 'sklearn' if name == 'scikit-learn' else name
        if module in sys.modules:
            loaded_changes.append(module)
if needed:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *needed])
if loaded_changes:
    raise RuntimeError('Packages were updated after import: restart the kernel and run from the top.')
print({name: importlib.metadata.version(name) for name in required})


{'numpy': '2.2.6', 'pandas': '2.2.3', 'scipy': '1.14.1', 'scikit-learn': '1.5.2', 'matplotlib': '3.9.2', 'joblib': '1.4.2'}


## Step 03: Import shared code and fix the research configuration

`SEED=17` controls group randomization; it is not a guarantee of identical GPU arithmetic.
`RUN_DIR` creates a new experiment directory and protects existing results. The code records the
environment rather than assuming that equal seeds prove identical outputs on every machine.

In [3]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import masheart
from masheart.data import (verify_sources, load_frame, split_development, assert_disjoint,
                           public_payload, compose_text)
from masheart.baseline import make_classifier, make_dummy_classifier
from masheart.metrics import evaluate, paired_cluster_bootstrap
from IPython.display import display, Image

assert Path(masheart.__file__).resolve().is_relative_to(PROJECT_ROOT)
DATA_DIR = PROJECT_ROOT / 'data'
CPU_CONFIG = json.loads((PROJECT_ROOT / 'configs' / 'cpu.json').read_text())
SEED = CPU_CONFIG['seed']
LABELS = ['NO', 'YS']
RUN_DIR = PROJECT_ROOT / 'runs' / ('notebook_' + str(time.time_ns()))
RUN_DIR.mkdir(parents=True, exist_ok=False)
np.random.seed(SEED)
print('New run:', RUN_DIR)


New run: C:\Users\huynh\OneDrive\Máy tính\ThsHUIT\HeartMAS\MASHeart\.verification\01_Data_Exploration_runtime\MASHeart_runtime_74675a45c3\runs\notebook_1794720698385749900


## Step 04: Verify data and read development only

`verify_sources()` checks manifest paths, byte counts and SHA-256 without parsing test labels.
`load_frame(..., 'dev')` reads UTF-8 JSONL and validates schema, ID uniqueness and labels.
Preserve question/text source strings for quoting. Do not remove negation, numbers or conditions.
The original release has no independent train set: this project uses educational development fitting.

In [4]:
source_audit = verify_sources(DATA_DIR)
dev = load_frame(DATA_DIR / 'raw' / 'a_dev.jsonl', 'dev')
assert len(dev) == 125 and dev.group_id.nunique() == 11
display(dev[['id', 'question', 'text', 'label', 'group_id']].head(3))
display(dev.label.value_counts().reindex(LABELS).rename('examples').to_frame())
print('Verified source files:', source_audit['files_verified'])


,id,question,text,label,group_id
0,mc-23435582-00,"In women with pre-eclampsia, does treatment wi...",Supplementation during pregnancy with a medica...,YS,qg-23435582
1,mc-23435582-01,"In women with pre-eclampsia, does treatment wi...",Oral L-arginine supplementation did not reduce...,NO,qg-23435582
2,mc-23435582-02,"In women with pre-eclampsia, does treatment wi...",L-Arginine load in pregnant women is associate...,NO,qg-23435582


,examples
label,
NO,41
YS,84


Verified source files: 5


## Step 05: Visualize label balance and statement lengths

**Scientific display:** export the measured table as well as PNG/SVG. The label chart exposes class
imbalance; the histogram describes whitespace length, which differs from the model's subword tokens.
These plots summarize development only and do not select examples based on test scores.

In [5]:
development_profile = dev[['id', 'group_id', 'label']].copy()
development_profile['whitespace_words'] = dev.text.map(lambda value: len(value.split()))
development_profile.to_csv(RUN_DIR / 'development_profile.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
dev.label.value_counts().reindex(LABELS).plot.bar(ax=axes[0], color=['#c56c39', '#287da8'])
axes[0].set(title='Development label balance', xlabel='Stance', ylabel='Examples')
axes[0].tick_params(axis='x', rotation=0)
axes[1].hist(development_profile.whitespace_words, bins=10, color='#287da8', edgecolor='white')
axes[1].set(title='Statement length', xlabel='Whitespace words', ylabel='Examples')
fig.tight_layout()
fig.savefig(RUN_DIR / 'development_overview.png', dpi=200, bbox_inches='tight')
fig.savefig(RUN_DIR / 'development_overview.svg', bbox_inches='tight')
plt.show()


C:\Users\huynh\AppData\Local\Temp\ipykernel_6760\2565811707.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 06: Create group-disjoint training and validation

`validation_fraction=0.25` means a fraction of source groups, not exactly 25% of rows.
Whole linked groups remain together. For seed 17, development becomes 95 examples/8 groups for
training and 30 examples/3 groups for validation. `assert_disjoint()` also checks IDs, PMIDs and
normalized statements. The official test is not an input to this splitting function.

In [6]:
train, validation = split_development(dev, SEED, CPU_CONFIG['validation_fraction'])
split_audit = assert_disjoint(train, validation)
split_summary = pd.DataFrame([
    {'split': name, 'n': len(frame), 'groups': frame.group_id.nunique(),
     'NO': int(frame.label.eq('NO').sum()), 'YS': int(frame.label.eq('YS').sum())}
    for name, frame in [('train', train), ('validation', validation)]
])
display(split_summary)
split_summary.to_csv(RUN_DIR / 'development_split.csv', index=False)
assert len(train) == 95 and len(validation) == 30


,split,n,groups,NO,YS
0,train,95,8,32,63
1,validation,30,3,9,21


## Step 07: Protect the inference feature boundary

`public_payload()` returns question and text only. ID/group/PMID may be used to audit or join results,
while gold labels are used only for fitting or scoring. `compose_text()` prepares the same two fields
for the CPU classifier. Changing gold or metadata must not change the public model input.

In [7]:
example = train.iloc[0].to_dict()
modified = dict(example, label='NO' if example['label'] == 'YS' else 'YS', pmid='EXCLUDED')
assert public_payload(example) == public_payload(modified)
display(public_payload(example))
print(compose_text(example))


{'question': 'In women with pre-eclampsia, does treatment with L Arginine, compared to placebo, reduce blood pressure or pre-eclampsia',
 'text': 'Supplementation during pregnancy with a medical food containing L-arginine and antioxidant vitamins reduced the incidence of pre-eclampsia in a population at high risk of the condition.'}

Question: In women with pre-eclampsia, does treatment with L Arginine, compared to placebo, reduce blood pressure or pre-eclampsia
Statement: Supplementation during pregnancy with a medical food containing L-arginine and antioxidant vitamins reduced the incidence of pre-eclampsia in a population at high risk of the condition.


## Step 08: Inspect source groups without opening test

Several statements can arise from a related source group. Count observations and classes by group;
this motivates group splitting and later cluster bootstrap instead of assuming rows are independent.

In [8]:
grouped = dev.groupby(['group_id', 'label']).size().unstack(fill_value=0).reindex(columns=LABELS)
grouped.to_csv(RUN_DIR / 'development_groups.csv')
grouped.plot.barh(stacked=True, figsize=(8, 5), color=['#c56c39', '#287da8'])
plt.xlabel('Statements'); plt.ylabel('Linked source group')
plt.title('Development-only group structure'); plt.tight_layout(); plt.show()


C:\Users\huynh\AppData\Local\Temp\ipykernel_6760\1441572523.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title('Development-only group structure'); plt.tight_layout(); plt.show()


## Step 09: Compare original text with an audit key

Audit normalization changes case/whitespace only; do not overwrite the source. Scope, negation,
population, time and numbers carry meaning. A token count is not a clinical evidence-quality score.

In [9]:
from masheart.data import normalize_for_audit
synthetic = '  Treatment did NOT change outcome after 2 days.  '
print('Source:', repr(synthetic))
print('Audit key:', normalize_for_audit(synthetic))
print('Development normalized duplicate rows:', dev.text.map(normalize_for_audit).duplicated().sum())
print('Prepared files:', sorted(p.name for p in RUN_DIR.iterdir()))


Source: '  Treatment did NOT change outcome after 2 days.  '
Audit key: treatment did not change outcome after 2 days.
Development normalized duplicate rows: 1
Prepared files: ['development_groups.csv', 'development_overview.png', 'development_overview.svg', 'development_profile.csv', 'development_split.csv']


## Continue to the full experiment
Open **MASHeart_Step_by_Step.ipynb** for actual CPU training and optional T4 QLoRA/MAS, frozen evaluation,
figures, uncertainty and exported artifacts. This exploratory notebook deliberately does not train
a model, read official test labels or present a performance estimate.